# Material + Reaction Rate - ENDF/B-VIII.1

This notebook compares YAMC and OpenMC **transport-coupled** material depletion
using the ENDF/B-VIII.1 nuclear data library.

## Model Description

- **Method**: transport-coupled depletion. A Monte Carlo neutron transport solve
  runs at every irradiation step; the flux-weighted one-group reaction rates it
  produces drive the burnup ODE. YAMC uses
  `Model.simulate_transmutation(method="coupled", ...)`; OpenMC uses
  `openmc.deplete.CoupledOperator` with a depletion integrator.
- **Difference from the flux-spectrum sibling**: the
  `material-flux-spectrum` notebook prescribes a fixed multigroup flux and never
  transports particles (`Material.transmute` / `material.deplete(multigroup_flux=...)`).
  Here the reaction rates are *computed from transport* against the actual
  geometry and source each step, so geometry, self-shielding and spectrum are
  modelled rather than assumed.
- **Geometry**: a solid sphere (vacuum boundary) with a 14.06 MeV point neutron
  source at the origin. The cell volume is computed by stochastic sampling.
- **Material**: a single seed nuclide at 1.0 g/cm3, 294 K, marked depletable.
- **Chain**: `chain_endf_b8.1_sfr.xml` for OpenMC; the cached YAMC arrow chain.
  Both reduced per nuclide to `reduce_chain_level=2`.
- **Timesteps**: 3 x 1 hour irradiation + 2 x 15 day cooling.
- **Comparison**: nuclide atom densities at each timestep.

In [ ]:
# Setup and imports
import time
import numpy as np
import pandas as pd
import glob
import os
import yamc
import yamc as mc
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

# Configuration
NUCLEAR_DATA_DIR = {
    "endf-b8.1": "/home/jon/nuclear_data/endf-b8.1-hdf5/endfb-viii.1-hdf5/",
}

NUCLEAR_DATA = "endf-b8.1"
mc.transport_data = NUCLEAR_DATA

# Codes/backends to run. Comma-separated list via the MC_BACKENDS env var.
# Tokens:
#   openmc-cpu  - reference OpenMC coupled depletion
#   yamc-cpu    - YAMC coupled depletion on CPU
#   yamc-gpu    - YAMC coupled depletion on GPU
# This notebook does a single YAMC run (no cpu/gpu split), so the tokens
# collapse to two booleans: RUN_OPENMC and RUN_YAMC. Any yamc-* token enables
# the YAMC run. Default runs everything. Examples:
#   MC_BACKENDS=openmc-cpu     # OpenMC only
#   MC_BACKENDS=yamc-cpu       # YAMC only
_VALID_BACKENDS = ("openmc-cpu", "yamc-cpu", "yamc-gpu")
_raw = os.environ.get("MC_BACKENDS", "openmc-cpu,yamc-cpu,yamc-gpu")
MC_BACKENDS = [b.strip().lower() for b in _raw.split(",") if b.strip()]
_bad = [b for b in MC_BACKENDS if b not in _VALID_BACKENDS]
if _bad:
    raise ValueError(
        f"MC_BACKENDS contains unknown token(s) {_bad}; "
        f"valid tokens are {list(_VALID_BACKENDS)}"
    )
if not MC_BACKENDS:
    raise ValueError("MC_BACKENDS resolved to an empty list")
RUN_OPENMC = "openmc-cpu" in MC_BACKENDS
RUN_YAMC = any(b.startswith("yamc-") for b in MC_BACKENDS)
print(f"Backends: {MC_BACKENDS}")

# Transmutation data: OpenMC reads the XML chain; YAMC reads the cached arrow
# directory (auto-downloaded into ~/.cache/yamc/transmutation-endf-b8.1-sfr.arrow/).
# The primary chain path matches the transmutation siblings; if it is absent we
# fall back to any openmc_data copy of the same SFR chain so the coupled OpenMC
# solve can run.
_CHAIN_CANDIDATES = [
    "/home/jon/yamc-org/openmc_data/src/openmc_data/depletion/chain_endf_b8.1_sfr.xml",
]
try:
    import openmc_data as _omc_data
    _CHAIN_CANDIDATES.append(
        os.path.join(os.path.dirname(_omc_data.__file__), "depletion", "chain_endf_b8.1_sfr.xml")
    )
except Exception:
    pass
# Any installed openmc_data package copy on this machine.
_CHAIN_CANDIDATES += sorted(glob.glob(
    "/home/jon/*/.venv/lib/python*/site-packages/openmc_data/depletion/chain_endf_b8.1_sfr.xml"
))
CHAIN_FILE_OPENMC = next((p for p in _CHAIN_CANDIDATES if os.path.exists(p)), _CHAIN_CANDIDATES[0])
TRANSMUTATION_DATA = os.path.expanduser("~/.cache/yamc/transmutation-endf-b8.1-sfr.arrow")

# Load the YAMC chain once; reduce(...) returns a fresh Chain per call,
# so the loaded object can be reused across nuclides without mutation.
_t0 = time.perf_counter()
YAMC_CHAIN = mc.TransmutationChain(TRANSMUTATION_DATA)
YAMC_CHAIN_LOAD_S = time.perf_counter() - _t0
print(f"YAMC chain load: {YAMC_CHAIN_LOAD_S:.3f}s ({len(YAMC_CHAIN)} nuclides)")

# Optional smoke-test cap on the iteration list (env var; unset/0 = full).
NUMBER_OF_SIMULATIONS = int(os.environ.get("YAMC_VERIFY_NUMBER_OF_SIMULATIONS", "0")) or None

# Transport / geometry parameters
SPHERE_RADIUS = 5.0          # cm (dense enough to give interactions)
SOURCE_ENERGY = 14.06e6      # eV (D-T fusion)
MATERIAL_DENSITY = 1.0       # g/cm3
TOTAL_PARTICLES = 100000     # transport histories per coupled step

# Depletion schedule (transport-coupled)
TRANSMUTE_TIMESTEPS = [3600.0, 3600.0, 3600.0, 15 * 86400.0, 15 * 86400.0]
# Source emission rate [n/s] held during each step; 0 = cooling (no transport).
TRANSMUTE_SOURCE_RATES = [1e18, 1e18, 1e18, 0.0, 0.0]
TRANSMUTE_PHASES = ["irrad", "irrad", "irrad", "cool", "cool"]
TRANSMUTE_REDUCE_LEVEL = 2

# Density threshold for comparison
DENSITY_THRESHOLD = 1e-15

In [ ]:
# Results cache - load reference/YAMC results from disk if a previous run
# produced them with the same parameters; otherwise run and write them out for
# next time. OpenMC reference results are keyed by run parameters only; YAMC
# results additionally fold yamc_fingerprint() into their metadata so the cache
# auto-invalidates when the installed yamc version or git commit changes.
import sys
from pathlib import Path as _Path

_repo_root = _Path.cwd().resolve()
while _repo_root != _repo_root.parent and not (_repo_root / "cache_utils.py").exists():
    _repo_root = _repo_root.parent
if (_repo_root / "cache_utils.py").exists() and str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))
from cache_utils import cached_or_run, yamc_fingerprint

CACHE_DIR = _Path("cache")
_COMMON_META = {
    "library": NUCLEAR_DATA,
    "method": "coupled",
    "chain_file": CHAIN_FILE_OPENMC,
    "sphere_radius": SPHERE_RADIUS,
    "source_energy": SOURCE_ENERGY,
    "material_density": MATERIAL_DENSITY,
    "timesteps": list(TRANSMUTE_TIMESTEPS),
    "source_rates": list(TRANSMUTE_SOURCE_RATES),
    "reduce_level": TRANSMUTE_REDUCE_LEVEL,
}
OPENMC_CACHE_META = {**_COMMON_META, "total_particles": TOTAL_PARTICLES}
# YAMC is stochastic and tracks the installed build, so fold both into its meta.
YAMC_CACHE_META = {**_COMMON_META, "total_particles": TOTAL_PARTICLES, **yamc_fingerprint()}

In [ ]:
def run_openmc_transmutation(nuclide):
    """Run OpenMC transport-coupled depletion for a single seed nuclide.

    Builds a sphere model with a 14.06 MeV point source, wraps it in a
    ``CoupledOperator`` (``normalization_mode='source-rate'`` so the prescribed
    ``source_rates`` are emission rates in n/s), and integrates the schedule with
    a predictor integrator. The depletion chain is reduced to the same level as
    YAMC so only reachable products carry cross-section data.

    Returns dict with reduce_s, deplete_s, time_s and a ``steps`` list of
    ``{nuclide: atom_density}`` dicts (one per timestep, skipping the initial
    state) to match the comparison helpers.
    """
    import tempfile
    import openmc
    import openmc.deplete

    openmc.reset_auto_ids()
    openmc.config['cross_sections'] = os.path.join(NUCLEAR_DATA_DIR[NUCLEAR_DATA], 'cross_sections.xml')
    openmc.config['chain_file'] = CHAIN_FILE_OPENMC

    material = openmc.Material(material_id=1, name=nuclide)
    material.add_nuclide(nuclide, 1.0, percent_type='ao')
    material.set_density('g/cm3', MATERIAL_DENSITY)
    material.temperature = 294.0
    material.depletable = True
    material.volume = 4.0 / 3.0 * np.pi * SPHERE_RADIUS ** 3

    sphere = openmc.Sphere(r=SPHERE_RADIUS, boundary_type='vacuum')
    cell = openmc.Cell(name='sphere', region=-sphere, fill=material)
    geometry = openmc.Geometry([cell])

    source = openmc.IndependentSource(
        space=openmc.stats.Point((0, 0, 0)),
        angle=openmc.stats.Isotropic(),
        energy=openmc.stats.Discrete([SOURCE_ENERGY], [1.0]),
    )
    settings = openmc.Settings(
        particles=TOTAL_PARTICLES,
        batches=10,
        source=source,
        seed=1,
        run_mode='fixed source',
        output={'tallies': False, 'summary': False},
    )
    model = openmc.model.Model(geometry, openmc.Materials([material]), settings)

    # Reduce the depletion chain to this nuclide's neighborhood (timed for parity).
    t0 = time.perf_counter()
    chain = openmc.deplete.Chain.from_xml(CHAIN_FILE_OPENMC)
    reduced = chain.reduce([nuclide], TRANSMUTE_REDUCE_LEVEL)
    reduce_s = time.perf_counter() - t0

    # Run inside a scratch directory so the statepoint / results files written by
    # the integrator do not litter the notebook folder.
    work = tempfile.mkdtemp(prefix="openmc_coupled_")
    results_h5 = os.path.join(work, "depletion_results.h5")
    reduced_xml = os.path.join(work, "reduced_chain.xml")
    reduced.export_to_xml(reduced_xml)

    cwd = os.getcwd()
    t1 = time.perf_counter()
    try:
        os.chdir(work)
        operator = openmc.deplete.CoupledOperator(
            model, chain_file=reduced_xml, normalization_mode='source-rate',
        )
        integrator = openmc.deplete.PredictorIntegrator(
            operator,
            timesteps=TRANSMUTE_TIMESTEPS,
            source_rates=TRANSMUTE_SOURCE_RATES,
            timestep_units='s',
        )
        integrator.integrate(path=results_h5)
    finally:
        os.chdir(cwd)
    deplete_s = time.perf_counter() - t1

    results = openmc.deplete.Results(results_h5)
    # Results hold n+1 step materials (index 0 = initial state); skip index 0.
    steps = []
    for step in range(1, len(results)):
        mat = results[step].get_material("1")
        steps.append(mat.get_nuclide_atom_densities())

    return {
        'reduce_s': reduce_s,
        'deplete_s': deplete_s,
        'time_s': reduce_s + deplete_s,
        'steps': steps,
    }

In [ ]:
def run_yamc_transmutation(nuclide):
    """Run YAMC transport-coupled depletion for a single seed nuclide.

    Builds the same sphere geometry with a 14.06 MeV point source, then calls
    ``Model.simulate_transmutation(method="coupled", ...)``, which re-runs
    transport at every irradiation step and feeds the flux-weighted reaction
    rates into the burnup ODE. Cooling steps (rate 0) become ``Cooldown`` entries
    that decay only. The chain is reduced via ``reduce_chain_level`` to match
    OpenMC's ``chain.reduce(...)``.

    Returns dict with reduce_s, deplete_s, time_s and a ``steps`` list of
    ``{nuclide: atom_density}`` dicts (one per timestep, skipping the initial
    state) to match the comparison helpers.
    """
    material = mc.Material(
        composition={nuclide: 1.0},
        density=MATERIAL_DENSITY,
        temperature=294,
        transmutable=True,
        id=1,
        name=nuclide,
    )
    material.read_nuclear_data(NUCLEAR_DATA)

    sphere = mc.Sphere(x0=0.0, y0=0.0, z0=0.0, radius=SPHERE_RADIUS, boundary='vacuum')
    cell = mc.Cell(id=1, name='sphere', region=sphere.below, material=material)
    geometry = mc.Geometry([cell])
    # Stochastic cell volume (sets both cell.volume and material.volume).
    geometry.calculate_volume(samples=1_000_000)

    source = mc.NeutronSource(
        position=(0, 0, 0),
        direction=mc.sources.Isotropic(),
        energy=mc.sources.Discrete([SOURCE_ENERGY], [1.0]),
    )
    model = mc.Model(
        geometry=geometry, source=source, total_particles=TOTAL_PARTICLES, seed=1,
    )

    # Irradiation pulses (rate in n/s) and cooling steps, matching OpenMC.
    schedule_steps = []
    for dt, rate in zip(TRANSMUTE_TIMESTEPS, TRANSMUTE_SOURCE_RATES):
        if rate > 0:
            schedule_steps.append(mc.Pulse(source=source, rate=rate, duration=dt))
        else:
            schedule_steps.append(mc.Cooldown(duration=dt))
    schedule = mc.PulseSchedule(schedule_steps)

    # reduce_chain_level is applied internally; time the reduction by measuring
    # the chain reduce separately for parity with the OpenMC breakdown.
    t0 = time.perf_counter()
    _ = YAMC_CHAIN.reduce([nuclide], level=TRANSMUTE_REDUCE_LEVEL)
    reduce_s = time.perf_counter() - t0

    t1 = time.perf_counter()
    results = model.simulate_transmutation(
        method="coupled",
        schedule=schedule,
        transmutation_file=TRANSMUTATION_DATA,
        reduce_chain_level=TRANSMUTE_REDUCE_LEVEL,
    )
    deplete_s = time.perf_counter() - t1

    # num_steps step materials (index 0 = initial state); skip index 0.
    steps = []
    for step in range(1, results.num_steps + 1):
        d = results.get_material_nuclides(material_id=1, step=step)
        steps.append(dict(d) if d else {})

    return {
        'reduce_s': reduce_s,
        'deplete_s': deplete_s,
        'time_s': reduce_s + deplete_s,
        'steps': steps,
    }

In [ ]:
def transmute_quality_label(max_rel_diff):
    """Assign quality label based on max relative difference (%) for transmutation."""
    if max_rel_diff < 0.1:
        return "Excellent"
    elif max_rel_diff < 1.0:
        return "Good"
    elif max_rel_diff < 5.0:
        return "Fair"
    else:
        return "Poor"


def compare_transmutation_steps(yamc_result, openmc_result):
    """Compare transmutation results step-by-step.

    Returns dict with detail_rows, summary metrics, and quality.
    """
    detail_rows = []
    shared_rel_diffs = []
    yamc_only_count = 0
    openmc_only_count = 0

    for step_idx in range(len(TRANSMUTE_TIMESTEPS)):
        ymc_step = yamc_result['steps'][step_idx] if step_idx < len(yamc_result['steps']) else {}
        omc_step = openmc_result['steps'][step_idx] if step_idx < len(openmc_result['steps']) else {}

        all_products = set(ymc_step.keys()) | set(omc_step.keys())

        for product in sorted(all_products):
            y = ymc_step.get(product, 0.0)
            o = omc_step.get(product, 0.0)
            max_val = max(abs(y), abs(o))
            if max_val < DENSITY_THRESHOLD:
                continue

            y_above = abs(y) >= DENSITY_THRESHOLD
            o_above = abs(o) >= DENSITY_THRESHOLD
            shared = y_above and o_above

            if max_val > 0:
                rd = abs(y - o) / max_val * 100.0
            else:
                rd = 0.0

            detail_rows.append({
                'product': product,
                'step': step_idx + 1,
                'phase': TRANSMUTE_PHASES[step_idx],
                'yamc': y,
                'openmc': o,
                'rel_diff_pct': rd,
                'shared': shared,
            })

            if shared:
                shared_rel_diffs.append(rd)
            elif y_above and not o_above:
                yamc_only_count += 1
            else:
                openmc_only_count += 1

    if shared_rel_diffs:
        max_rel_diff_pct = max(shared_rel_diffs)
        rms_rel_diff_pct = float(np.sqrt(np.mean(np.array(shared_rel_diffs) ** 2)))
    else:
        max_rel_diff_pct = 0.0
        rms_rel_diff_pct = 0.0

    n_shared = len({dr['product'] for dr in detail_rows if dr['shared']})
    n_total = len({dr['product'] for dr in detail_rows})

    quality = transmute_quality_label(max_rel_diff_pct)
    speedup = openmc_result['time_s'] / yamc_result['time_s'] if yamc_result['time_s'] > 0 else float('inf')

    return {
        'detail_rows': detail_rows,
        'max_rel_diff_pct': max_rel_diff_pct,
        'rms_rel_diff_pct': rms_rel_diff_pct,
        'n_shared': n_shared,
        'n_total': n_total,
        'yamc_only_count': yamc_only_count,
        'openmc_only_count': openmc_only_count,
        'quality': quality,
        'speedup': speedup,
    }


def create_density_comparison_table(detail_rows, step_idx):
    """Create a density comparison table for a specific timestep."""
    step_rows = [dr for dr in detail_rows if dr['step'] == step_idx + 1]

    rows = []
    for dr in sorted(step_rows, key=lambda x: -max(abs(x['yamc']), abs(x['openmc']))):
        rows.append({
            'Nuclide': dr['product'],
            'YAMC': f"{dr['yamc']:.6e}",
            'OpenMC': f"{dr['openmc']:.6e}",
            'Rel. Diff (%)': f"{dr['rel_diff_pct']:.4f}%",
            'Status': 'Shared' if dr['shared'] else ('YAMC only' if abs(dr['yamc']) >= DENSITY_THRESHOLD else 'OpenMC only'),
        })

    return pd.DataFrame(rows)


def create_density_evolution_plot(detail_rows, nuclide):
    """Density evolution of the top products with YAMC and OpenMC on one shared axis.

    Each product gets a single colour; YAMC is drawn as a solid line with round
    markers and OpenMC as a dashed line with x markers, so the two codes overlay
    for a direct comparison instead of sitting in separate side-by-side panels.
    """

    # Get all products and their max densities
    product_max = {}
    for dr in detail_rows:
        p = dr['product']
        val = max(abs(dr['yamc']), abs(dr['openmc']))
        if p not in product_max or val > product_max[p]:
            product_max[p] = val

    # Top 10 products by max density (excluding the starting nuclide itself)
    top_products = sorted(
        [(p, v) for p, v in product_max.items() if p != nuclide],
        key=lambda x: -x[1]
    )[:10]

    if not top_products:
        return None

    steps = list(range(1, len(TRANSMUTE_TIMESTEPS) + 1))
    step_labels = [f"Step {s}\n({TRANSMUTE_PHASES[s-1]})" for s in steps]

    colors = [
        '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
        '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf'
    ]

    fig = go.Figure()

    for idx, (product, _) in enumerate(top_products):
        yamc_densities = []
        openmc_densities = []
        for step in steps:
            ymc_val = 0.0
            omc_val = 0.0
            for dr in detail_rows:
                if dr['product'] == product and dr['step'] == step:
                    ymc_val = dr['yamc']
                    omc_val = dr['openmc']
                    break
            yamc_densities.append(ymc_val)
            openmc_densities.append(omc_val)

        color = colors[idx % len(colors)]

        fig.add_trace(go.Scatter(
            x=steps, y=yamc_densities, mode='lines+markers',
            name=f"yamc-{product}", legendgroup=product,
            line=dict(color=color),
            marker=dict(color=color, symbol='circle', size=6),
        ))
        fig.add_trace(go.Scatter(
            x=steps, y=openmc_densities, mode='lines+markers',
            name=f"openmc-{product}", legendgroup=product,
            line=dict(color=color, dash='dash'),
            marker=dict(color=color, symbol='x', size=7),
        ))

    fig.update_xaxes(title_text="Timestep", tickvals=steps, ticktext=step_labels)
    fig.update_yaxes(type="log", title_text="Atom Density (atoms/b-cm)", exponentformat='e')

    fig.update_layout(
        height=550,
        title=f"{nuclide} - Product Density Evolution (Top 10), solid=YAMC dashed=OpenMC",
        legend=dict(x=1.02, y=1.0),
        margin=dict(l=60, r=170, t=60, b=60),
    )

    return fig


def create_quality_metrics_table(comparison):
    """Create a summary table of quality metrics."""
    rows = [
        {'Metric': 'Max Rel. Diff (shared)', 'Value': f"{comparison['max_rel_diff_pct']:.4f}%"},
        {'Metric': 'RMS Rel. Diff (shared)', 'Value': f"{comparison['rms_rel_diff_pct']:.4f}%"},
        {'Metric': 'Shared Products', 'Value': str(comparison['n_shared'])},
        {'Metric': 'Total Products', 'Value': str(comparison['n_total'])},
        {'Metric': 'YAMC-only entries', 'Value': str(comparison['yamc_only_count'])},
        {'Metric': 'OpenMC-only entries', 'Value': str(comparison['openmc_only_count'])},
        {'Metric': 'Quality', 'Value': comparison['quality']},
        {'Metric': 'Speedup', 'Value': f"{comparison['speedup']:.1f}x"},
    ]
    return pd.DataFrame(rows)

In [ ]:
# Discover nuclides: stable neutron nuclides that are in the chain.
# Restricted to stable until YAMC issue #45 is fixed (reduce() pulls in
# fission-yield products which triggers ~1000 cross-section downloads).
#
# When OpenMC runs, the OpenMC chain is the source of truth (OpenMC needs it
# for the coupled solve) and nuclides come from the HDF5 dir. When OpenMC is
# excluded, the OpenMC chain may be absent, so discover stable nuclides from
# the already-loaded YAMC chain (stable = present in the chain with no
# half-life entry) and source names from the local yamc Arrow cache.
import re
from pathlib import Path as _P

neutron_dir = os.path.join(NUCLEAR_DATA_DIR[NUCLEAR_DATA], 'neutron')

if RUN_OPENMC:
    import openmc.deplete

    _t0 = time.perf_counter()
    chain = openmc.deplete.Chain.from_xml(CHAIN_FILE_OPENMC)
    OPENMC_CHAIN_LOAD_S = time.perf_counter() - _t0
    print(f"OpenMC chain load: {OPENMC_CHAIN_LOAD_S:.3f}s ({len(chain.nuclides)} nuclides)")

    chain_stable_names = {nuc.name for nuc in chain.nuclides if nuc.half_life is None}
    print(f"({len(chain_stable_names)} stable)")

    h5_files = sorted(glob.glob(os.path.join(neutron_dir, '*.h5')))
    nuclides = []
    for f in h5_files:
        name = os.path.splitext(os.path.basename(f))[0]
        if '_m' in name:  # skip metastable
            continue
        if name in chain_stable_names:
            nuclides.append(name)

    if not nuclides:
        raise FileNotFoundError(
            f"No stable nuclides found in chain from {neutron_dir}\n"
            f"Please check that the chain file and nuclear data directory exist."
        )
else:
    OPENMC_CHAIN_LOAD_S = 0.0
    # Stable in the YAMC chain: present in nuclide_names with no half-life entry.
    _hl = YAMC_CHAIN.half_lives
    chain_stable_names = {
        n for n in YAMC_CHAIN.nuclide_names if '_m' not in n and _hl.get(n) is None
    }
    print(f"({len(chain_stable_names)} stable in YAMC chain)")

    yamc_cache = _P.home() / ".cache" / "yamc"
    prefix = f"{NUCLEAR_DATA}-"
    nuclides = []
    for p in sorted(yamc_cache.glob(f"{prefix}*.arrow")):
        if not p.is_dir():
            continue
        name = p.name[len(prefix):-len(".arrow")]
        if '_m' in name:  # skip metastable
            continue
        if not re.fullmatch(r"[A-Z][a-z]?\d+", name):
            continue
        if name in chain_stable_names:
            nuclides.append(name)

    if not nuclides:
        raise FileNotFoundError(
            f"No stable nuclides found in YAMC chain / cache at {yamc_cache}.\n"
            f"Run yamc once on any stable nuclide to populate the cache."
        )

if NUMBER_OF_SIMULATIONS:
    nuclides = nuclides[:NUMBER_OF_SIMULATIONS]

print(f"Running transport-coupled depletion for {len(nuclides)} nuclides...")
for nuc in nuclides:
    print(f"  {nuc}")

In [ ]:
# Run all transport-coupled depletions and collect results.
#
# Both codes are stochastic transport solves, so each is cached: OpenMC keyed by
# run parameters, YAMC additionally keyed by the installed yamc fingerprint.
all_results = {}

for nuclide in nuclides:
    print(f"\nProcessing {nuclide}...")
    per_nuclide = {}

    if RUN_YAMC:
        try:
            print("  Running YAMC coupled depletion...")
            yamc_result = cached_or_run(
                CACHE_DIR, NUCLEAR_DATA, "yamc", nuclide,
                lambda n=nuclide: run_yamc_transmutation(n),
                meta=YAMC_CACHE_META,
            )
            per_nuclide['yamc'] = yamc_result
            print(f"  YAMC: reduce={yamc_result['reduce_s']:.3f}s "
                  f"deplete={yamc_result['deplete_s']:.3f}s "
                  f"total={yamc_result['time_s']:.3f}s")
        except Exception as e:
            print(f"  YAMC Error: {e}")
            continue

    if RUN_OPENMC:
        try:
            print("  Running OpenMC coupled depletion...")
            openmc_result = cached_or_run(
                CACHE_DIR, NUCLEAR_DATA, "openmc", nuclide,
                lambda n=nuclide: run_openmc_transmutation(n),
                meta=OPENMC_CACHE_META,
            )
            per_nuclide['openmc'] = openmc_result
            print(f"  OpenMC: reduce={openmc_result['reduce_s']:.3f}s "
                  f"deplete={openmc_result['deplete_s']:.3f}s "
                  f"total={openmc_result['time_s']:.3f}s")
        except Exception as e:
            print(f"  OpenMC Error: {e}")
            continue

    # Compare only when both backends ran.
    if 'yamc' in per_nuclide and 'openmc' in per_nuclide:
        comparison = compare_transmutation_steps(per_nuclide['yamc'], per_nuclide['openmc'])
        per_nuclide['comparison'] = comparison
        print(f"  max_rel_diff={comparison['max_rel_diff_pct']:.4f}%, "
              f"quality={comparison['quality']}, "
              f"deplete-only speedup="
              f"{per_nuclide['openmc']['deplete_s']/per_nuclide['yamc']['deplete_s']:.1f}x")

    all_results[nuclide] = per_nuclide

print(f"\nCompleted {len(all_results)} / {len(nuclides)} nuclides successfully.")

## Results by Nuclide

Below are the comparison results for each nuclide. Each section includes:

- Density evolution plots (top 10 product nuclides)
- Quality metrics (max/RMS relative difference on shared products)
- Density comparison tables per timestep

In [ ]:
# Show chain-load timings once for the whole notebook
display(Markdown(
    f"**Chain load** (one-time, before any nuclide):  \n"
    f"OpenMC: {OPENMC_CHAIN_LOAD_S:.3f}s | YAMC: {YAMC_CHAIN_LOAD_S:.3f}s"
))

# Display results for each nuclide
for nuclide, results in all_results.items():
    yr = results.get('yamc')        # may be None when no yamc-* in MC_BACKENDS
    or_ = results.get('openmc')     # may be None when openmc-cpu not in MC_BACKENDS
    comparison = results.get('comparison')  # only present when both ran

    # Header
    display(Markdown(f"## {nuclide}"))

    # Performance comparison (deplete only - chain reduce is sub-millisecond)
    perf_parts = []
    if or_ is not None:
        perf_parts.append(f"OpenMC: {or_['deplete_s']:.3f}s")
    if yr is not None:
        perf_parts.append(f"YAMC: {yr['deplete_s']:.3f}s")
    if or_ is not None and yr is not None and yr['deplete_s'] > 0:
        perf_parts.append(f"Speedup: {or_['deplete_s'] / yr['deplete_s']:.1f}x")
    display(Markdown(f"**Performance:** " + " | ".join(perf_parts)))

    # Quality metrics and density tables compare OpenMC vs YAMC; without both
    # there is nothing to compare, so note it and continue rather than KeyError.
    if comparison is None:
        only = "OpenMC" if or_ is not None else "YAMC"
        display(Markdown(
            f"*Only {only} ran (MC_BACKENDS={MC_BACKENDS}); "
            f"comparison metrics and density tables are skipped.*"
        ))
        print()
        continue

    # Quality metrics table
    display(Markdown("### Quality Metrics"))
    metrics_table = create_quality_metrics_table(comparison)
    display(metrics_table.style.hide(axis='index'))

    # Density evolution plot
    fig = create_density_evolution_plot(comparison['detail_rows'], nuclide)
    if fig is not None:
        fig.show()

    # Density comparison tables per timestep
    for step_idx in range(len(TRANSMUTE_TIMESTEPS)):
        phase = TRANSMUTE_PHASES[step_idx]
        display(Markdown(f"### Step {step_idx + 1} ({phase})"))
        table = create_density_comparison_table(comparison['detail_rows'], step_idx)
        if not table.empty:
            display(table.style.hide(axis='index'))
        else:
            display(Markdown("*No products above density threshold.*"))

    print()  # Add spacing between nuclides